In [1]:
import pandas as pd
import numpy as np
import argparse
import sys, os
import subprocess
from pathlib import Path
#import logging
#import line_profiler
sys.path.append('/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM')

from src.preprocess import (strip_pdb,
                            create_dataframe, 
                            extract_chains_chothia,
                            fix_case_if_reused,
                            build_interface_pair,
                            get_shard_atoms_for_ligand)

In [2]:
BASE_DIR = Path("/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/negativeSamples")

pdbpath = BASE_DIR/ Path("test") / "9mnx.pdb"

In [3]:
pdbpath.stem

'9mnx'

In [3]:
#extract chains
try:
    chain_pairs = extract_chains_chothia(pdbpath)
    if not chain_pairs:
        raise ValueError(f"No chain pairs returned for {pdbpath}")
    ligand_chains, receptor_chains = chain_pairs[0]


    ### Some chains are annotated with int like "2" in sabdab. This leads to unphysical brooms downstream
    if not all(isinstance(v, str)and  v.isalpha() for v in ligand_chains):
        raise ValueError(f"Invalid characters in ligand chains: {ligand_chains}")

    if not all(isinstance(v, str)and  v.isalpha() for v in receptor_chains):
        raise ValueError(f"Invalid characters in receptor chains: {receptor_chains}")
    
    receptor_chains = fix_case_if_reused(ligand_chains, receptor_chains, Path(pdbpath).stem) # some chothia pdbs have wrong headers


except Exception as e:
    print(f"[ERROR] Problem processing file: {pdbpath}")
    print(f"Reason: {e}")
    exit(1)  # Non-zero exit code so subprocess.run will detect failure


#Build full complex_data DataFrame, strips HETATMs
complex_data = create_dataframe(strip_pdb(pdbpath))


In [5]:
chain_pairs[0]


(['F'], ['D', 'E'])

In [4]:

#Filter atoms in complex_data for interface, map atoms of receptor L -> R
lig2rec_ligand_data, lig2rec_receptor_data = build_interface_pair(
                                              complex_data=complex_data,
                                              ligand_chains=ligand_chains,
                                              receptor_chains=receptor_chains,
                                              ligand_depth=30,
                                              receptor_depth=30
                                              )



/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/src/preprocess.py:286: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  receptor_data["target atom_id"] = receptor_data.apply(


In [7]:
lig2rec_ligand_data["chain_id"].unique()



array(['F'], dtype=object)

In [8]:
lig2rec_receptor_data["chain_id"].unique()

array(['D', 'E'], dtype=object)

In [9]:
# Drop the 'target atom_id' column
lig2rec_receptor_data = lig2rec_receptor_data.drop(columns=['target atom_id'])

In [5]:

# Concatenate them
combined_df = pd.concat([lig2rec_ligand_data, lig2rec_receptor_data], ignore_index=True)




In [6]:
combined_df


,record_type,atom_number,atom_name,res_name,chain_id,res_num,x,y,z,occupancy,temp_factor,element,target atom_id
0,ATOM,1807,N,ASN,F,372,150.153,104.687,188.898,1.0,50.29,N,NaN
1,ATOM,1808,CA,ASN,F,372,149.384,105.736,189.528,1.0,50.29,C,NaN
2,ATOM,1809,C,ASN,F,372,148.342,106.345,188.623,1.0,50.29,C,NaN
3,ATOM,1810,O,ASN,F,372,147.808,107.382,188.958,1.0,50.29,O,NaN
4,ATOM,1811,CB,ASN,F,372,150.273,106.861,189.987,1.0,50.29,C,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1877,ATOM,5198,CA,GLY,E,100,134.668,145.476,186.297,1.0,38.98,C,1.0
1878,ATOM,5199,C,GLY,E,100,134.210,146.420,187.382,1.0,38.98,C,4.0
1879,ATOM,5200,O,GLY,E,100,134.938,147.331,187.746,1.0,38.98,O,20.0
1880,ATOM,5201,N,GLY,E,101,133.014,146.247,187.889,1.0,40.42,N,3.0


In [7]:
def pdb_line(row):
    return (
        f"{row['record_type']:<6}"
        f"{int(row['atom_number']):>5} "
        f"{row['atom_name']:>4} "
        f"{row['res_name']:>3} "
        f"{row['chain_id']:>1}"
        f"{int(row['res_num']):>4}    "
        f"{float(row['x']):>8.3f}"
        f"{float(row['y']):>8.3f}"
        f"{float(row['z']):>8.3f}"
        f"{float(row['occupancy']):>6.2f}"
        f"{float(row['temp_factor']):>6.2f}          "
        f"{row['element']:>2}"
    )



def save_pdb(df, out_path):

    # Generate all lines
    pdb_lines = df.apply(pdb_line, axis=1)

    # Write to a pdb file
    with open(out_path, 'w') as f:
        for line in pdb_lines:
            f.write(line + '\n')


In [24]:
type(pdbpath.stem)

str

In [8]:
inter_pdb = BASE_DIR / "fpocket" / "interfacePDB"/ (pdbpath.stem + "_inter.pdb")

save_pdb(combined_df, inter_pdb)

In [15]:
inter_pdb.parent

PosixPath('/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/negativeSamples/fpocket/interfacePDB')

In [9]:

in_dir = inter_pdb.parent


In [10]:

inter_pdb.name

'9mnx_inter.pdb'

In [11]:
fp_results_dir = f"{inter_pdb.stem}_out" 


In [21]:

# run fpocket
m = 3
M = 6
i = 25

def run_fpocket(interface_pdb, in_dir, out_dir,fp_results_dir):
    subprocess.run([
        "conda", "run", "-n", "fpocket-env",
        "fpocket",  "-f", f"{interface_pdb}",
                    "-m", f"{m}",
                    "-M", f"{M}",
                    "-i", f"{i}"],
        cwd = in_dir,
        check=True)

    subprocess.run(['mv', f"{fp_results_dir}", f"{out_dir}"],
                   cwd = in_dir,
                   check=True)

In [13]:
fp_outout = Path("/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/negativeSamples/fpocket/output")

In [22]:
run_fpocket(inter_pdb.name, in_dir, fp_outout ,fp_results_dir)

***** POCKET HUNTING BEGINS ***** 
***** POCKET HUNTING ENDS ***** 

